# Irodori-TTS Colab

Run Irodori-TTS on Google Colab. Select a GPU runtime first, then run cells from top to bottom.

For long-text splitting and ChatGPT emoji annotation, set `REPO_URL` to your fork that includes these changes.

In [ ]:
# GPU check
!nvidia-smi

In [ ]:
# Settings
REPO_URL = "https://github.com/ikehiro79/Irodori-TTS.git"
BRANCH = "main"
APP_MODE = "normal"  # "normal" or "voicedesign"
HF_TOKEN = ""       # only needed for private Hugging Face models
USE_EXISTING_REPO = False

In [ ]:
# Clone or refresh repository
from pathlib import Path

repo_dir = Path("/content/Irodori-TTS")
if USE_EXISTING_REPO:
    if not repo_dir.exists():
        raise FileNotFoundError("/content/Irodori-TTS was not found.")
else:
    if repo_dir.exists():
        %cd /content/Irodori-TTS
        !git remote set-url origin {REPO_URL}
        !git fetch origin {BRANCH}
        !git checkout {BRANCH}
        !git reset --hard origin/{BRANCH}
        !git clean -fd
    else:
        %cd /content
        !git clone --branch {BRANCH} {REPO_URL} Irodori-TTS

%cd /content/Irodori-TTS
!git remote -v
!git status --short
!grep -R "Long Text Chunk Chars" -n /content/Irodori-TTS || true
!grep -R "ChatGPT" -n /content/Irodori-TTS/gradio_app.py /content/Irodori-TTS/gradio_app_voicedesign.py || true

In [ ]:
# Colab / Gradio compatibility patch
from pathlib import Path

for file in ["gradio_app.py", "gradio_app_voicedesign.py"]:
    path = Path(file)
    if not path.exists():
        continue
    text = path.read_text(encoding="utf-8")
    text = text.replace(
        'with gr.Blocks(title="Irodori-TTS Gradio") as demo:',
        'with gr.Blocks(title="Irodori-TTS Gradio", css=EMOJI_PALETTE_CSS) as demo:',
    )
    text = text.replace(
        'with gr.Blocks(title="Irodori-TTS VoiceDesign Gradio") as demo:',
        'with gr.Blocks(title="Irodori-TTS VoiceDesign Gradio", css=EMOJI_PALETTE_CSS) as demo:',
    )
    text = text.replace("        css=EMOJI_PALETTE_CSS,\n", "")
    path.write_text(text, encoding="utf-8")

print("patched Gradio css compatibility")

In [ ]:
# Hugging Face login. Public models do not need this.
if HF_TOKEN.strip():
    !huggingface-cli login --token {HF_TOKEN}
else:
    print("HF_TOKEN is empty; skipping Hugging Face login.")

In [ ]:
# Install dependencies
# sentencepiece<0.2 can fail on Colab Python, so install 0.2.0 first and remove the old constraint.
!python -m pip install -U pip setuptools wheel
!python -m pip install torch torchaudio --index-url https://download.pytorch.org/whl/cu121
!python -m pip install sentencepiece==0.2.0
!grep -v '^sentencepiece' requirements.txt > requirements_colab.txt
!python -m pip install -r requirements_colab.txt

In [ ]:
# Verify install
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

In [ ]:
# Load OpenAI API key for ChatGPT emoji annotation
# Save OPENAI_API_KEY in Colab Secrets first.
import os

if not os.environ.get("OPENAI_API_KEY"):
    try:
        from google.colab import userdata
        key = userdata.get("OPENAI_API_KEY")
        if key:
            os.environ["OPENAI_API_KEY"] = key
    except Exception as exc:
        print(f"Could not read Colab secret OPENAI_API_KEY: {exc}")

print("OPENAI_API_KEY loaded:", bool(os.environ.get("OPENAI_API_KEY")))

In [ ]:
# Launch Gradio UI
# Open the https://xxxxx.gradio.live URL printed in the log.
import shlex

if APP_MODE.lower().strip() == "voicedesign":
    app_file = "gradio_app_voicedesign.py"
    port = 7861
else:
    app_file = "gradio_app.py"
    port = 7860

cmd = f"python {shlex.quote(app_file)} --server-name 0.0.0.0 --server-port {port} --share"
print(cmd)
!{cmd}

## Notes

- `Long Text Chunk Chars (0=off)` default is `30`.
- Newlines always split text into separate chunks.
- For ChatGPT emoji annotation, add `OPENAI_API_KEY` to Colab Secrets and run the OpenAI key cell before launching Gradio.
- In Gradio, use `ChatGPTで絵文字を自動追加` to rewrite the text, or enable `Generate前にChatGPTで絵文字を自動追加` before generation.